# GPT-OSS-20B Demo

This notebook loads OpenAI's [GPT-OSS-20B](https://huggingface.co/openai/gpt-oss-20b) into TransformerLens for mechanistic interpretability.

GPT-OSS-20B is a Mixture of Experts (MoE) model with:
- 24 layers, d_model=2880, 32 experts, 4 experts per token
- MXFP4 quantized weights on HuggingFace (dequantized to BF16 during loading)
- Custom GLU activation and post-top-k softmax routing

**Memory:** The full model is ~40GB in BF16. This notebook loads directly from safetensors, bypassing the HuggingFace model pipeline to keep peak memory manageable. Use `N_LAYERS` to load fewer layers if needed.

**Requirements:** `transformers`, `safetensors`, `einops`, `psutil`

In [ ]:
import gc
import json
from pathlib import Path

import einops
import torch
from safetensors import safe_open
from transformers import AutoTokenizer
from transformers.integrations.mxfp4 import convert_moe_packed_tensors

from transformer_lens import HookedTransformer
from transformer_lens.HookedTransformerConfig import HookedTransformerConfig

## Configuration

Set `N_LAYERS` to control how many layers to load. Each layer is ~1.6GB. Use fewer layers to save memory.

In [ ]:
N_LAYERS = 24  # Full model. Set to 3-6 for quick testing.

## Model Loading Utilities

In [ ]:
def get_model_path():
    """Get the cached model path, downloading if necessary."""
    cache_path = Path.home() / ".cache/huggingface/hub/models--openai--gpt-oss-20b"
    snapshots = cache_path / "snapshots"

    if snapshots.exists():
        snapshot_dirs = list(snapshots.iterdir())
        if snapshot_dirs:
            return snapshot_dirs[0]

    print("Model not found in cache. Downloading...")
    from huggingface_hub import snapshot_download

    return Path(snapshot_download("openai/gpt-oss-20b"))


def create_config(n_layers=24):
    """Create TransformerLens config for GPT-OSS-20B."""
    return HookedTransformerConfig(
        n_layers=n_layers,
        d_model=2880,
        d_head=64,
        n_heads=64,
        d_mlp=2880,
        n_ctx=4096,
        d_vocab=201088,
        act_fn="silu",
        normalization_type="RMS",
        positional_embedding_type="rotary",
        rotary_base=150000,
        eps=1e-5,
        n_key_value_heads=8,
        gated_mlp=True,
        use_local_attn=False,
        rotary_dim=64,
        num_experts=32,
        experts_per_token=4,
        dtype=torch.bfloat16,
        device="cpu",
        original_architecture="GptOssForCausalLM",
        model_name="openai/gpt-oss-20b",
    )


_open_files = {}


def _get_tensor(hf_name, wmap, model_path):
    """Load a single tensor from the correct safetensors shard."""
    st_file = wmap[hf_name]
    filepath = str(model_path / st_file)
    if filepath not in _open_files:
        _open_files[filepath] = safe_open(filepath, framework="pt", device="cpu")
    return _open_files[filepath].get_tensor(hf_name)


def load_layer_weights(l, cfg, index, model_path):
    """Load and convert weights for one transformer layer from safetensors."""
    state_dict = {}
    wmap = index["weight_map"]
    prefix = f"model.layers.{l}"

    def gt(name):
        return _get_tensor(name, wmap, model_path)

    # LayerNorms
    state_dict[f"blocks.{l}.ln1.w"] = gt(f"{prefix}.input_layernorm.weight")
    state_dict[f"blocks.{l}.ln2.w"] = gt(f"{prefix}.post_attention_layernorm.weight")

    # Attention weights
    q_w = gt(f"{prefix}.self_attn.q_proj.weight")
    k_w = gt(f"{prefix}.self_attn.k_proj.weight")
    v_w = gt(f"{prefix}.self_attn.v_proj.weight")
    o_w = gt(f"{prefix}.self_attn.o_proj.weight")

    state_dict[f"blocks.{l}.attn.W_Q"] = einops.rearrange(q_w, "(n h) m -> n m h", n=cfg.n_heads)
    state_dict[f"blocks.{l}.attn._W_K"] = einops.rearrange(
        k_w, "(n h) m -> n m h", n=cfg.n_key_value_heads
    )
    state_dict[f"blocks.{l}.attn._W_V"] = einops.rearrange(
        v_w, "(n h) m -> n m h", n=cfg.n_key_value_heads
    )
    state_dict[f"blocks.{l}.attn.W_O"] = einops.rearrange(
        o_w, "m (n h) -> n h m", n=cfg.n_heads
    )
    del q_w, k_w, v_w, o_w

    # Attention biases
    q_bias_key = f"{prefix}.self_attn.q_proj.bias"
    if q_bias_key in wmap:
        state_dict[f"blocks.{l}.attn.b_Q"] = einops.rearrange(
            gt(q_bias_key), "(n h) -> n h", n=cfg.n_heads
        )
        state_dict[f"blocks.{l}.attn._b_K"] = einops.rearrange(
            gt(f"{prefix}.self_attn.k_proj.bias"), "(n h) -> n h", n=cfg.n_key_value_heads
        )
        state_dict[f"blocks.{l}.attn._b_V"] = einops.rearrange(
            gt(f"{prefix}.self_attn.v_proj.bias"), "(n h) -> n h", n=cfg.n_key_value_heads
        )
    else:
        state_dict[f"blocks.{l}.attn.b_Q"] = torch.zeros(cfg.n_heads, cfg.d_head, dtype=cfg.dtype)
        state_dict[f"blocks.{l}.attn._b_K"] = torch.zeros(
            cfg.n_key_value_heads, cfg.d_head, dtype=cfg.dtype
        )
        state_dict[f"blocks.{l}.attn._b_V"] = torch.zeros(
            cfg.n_key_value_heads, cfg.d_head, dtype=cfg.dtype
        )

    o_bias_key = f"{prefix}.self_attn.o_proj.bias"
    if o_bias_key in wmap:
        state_dict[f"blocks.{l}.attn.b_O"] = gt(o_bias_key)
    else:
        state_dict[f"blocks.{l}.attn.b_O"] = torch.zeros(cfg.d_model, dtype=cfg.dtype)

    # Router
    state_dict[f"blocks.{l}.mlp.W_gate.weight"] = gt(f"{prefix}.mlp.router.weight")
    state_dict[f"blocks.{l}.mlp.W_gate.bias"] = gt(f"{prefix}.mlp.router.bias")

    # Expert weights - dequantize MXFP4 to BF16
    gate_up_blocks = gt(f"{prefix}.mlp.experts.gate_up_proj_blocks")
    gate_up_scales = gt(f"{prefix}.mlp.experts.gate_up_proj_scales")
    gate_up_bias = gt(f"{prefix}.mlp.experts.gate_up_proj_bias")

    print(f"  Dequantizing layer {l} gate_up_proj...", end="", flush=True)
    gate_up_proj = convert_moe_packed_tensors(gate_up_blocks, gate_up_scales)
    del gate_up_blocks, gate_up_scales
    print(" done")

    down_blocks = gt(f"{prefix}.mlp.experts.down_proj_blocks")
    down_scales = gt(f"{prefix}.mlp.experts.down_proj_scales")
    down_bias = gt(f"{prefix}.mlp.experts.down_proj_bias")

    print(f"  Dequantizing layer {l} down_proj...", end="", flush=True)
    down_proj = convert_moe_packed_tensors(down_blocks, down_scales)
    del down_blocks, down_scales
    print(" done")

    # Split merged expert tensors into per-expert weights
    # Even columns -> gate, Odd columns -> up
    for e in range(cfg.num_experts):
        state_dict[f"blocks.{l}.mlp.experts.{e}.W_gate.weight"] = gate_up_proj[
            e, :, ::2
        ].T.contiguous()
        state_dict[f"blocks.{l}.mlp.experts.{e}.W_gate.bias"] = gate_up_bias[
            e, ::2
        ].contiguous()
        state_dict[f"blocks.{l}.mlp.experts.{e}.W_in.weight"] = gate_up_proj[
            e, :, 1::2
        ].T.contiguous()
        state_dict[f"blocks.{l}.mlp.experts.{e}.W_in.bias"] = gate_up_bias[
            e, 1::2
        ].contiguous()
        state_dict[f"blocks.{l}.mlp.experts.{e}.W_out.weight"] = down_proj[e].T.contiguous()
        state_dict[f"blocks.{l}.mlp.experts.{e}.W_out.bias"] = down_bias[e].contiguous()

    del gate_up_proj, gate_up_bias, down_proj, down_bias
    return state_dict

## Load Model

In [ ]:
model_path = get_model_path()
print(f"Model path: {model_path}")

with open(model_path / "model.safetensors.index.json") as f:
    index = json.load(f)

cfg = create_config(n_layers=N_LAYERS)
tokenizer = AutoTokenizer.from_pretrained(str(model_path))
model = HookedTransformer(cfg, tokenizer, move_to_device=False)

# Load embeddings
wmap = index["weight_map"]
model.load_state_dict(
    {"embed.W_E": _get_tensor("model.embed_tokens.weight", wmap, model_path)},
    strict=False,
)
gc.collect()

# Load layers one at a time
for l in range(N_LAYERS):
    print(f"Loading layer {l}/{N_LAYERS-1}...")
    layer_dict = load_layer_weights(l, cfg, index, model_path)
    for key in list(layer_dict.keys()):
        model.load_state_dict({key: layer_dict[key]}, strict=False)
        del layer_dict[key]
    del layer_dict
    gc.collect()

# Load final LayerNorm and unembed
model.load_state_dict(
    {"ln_final.w": _get_tensor("model.norm.weight", wmap, model_path)},
    strict=False,
)
model.load_state_dict(
    {"unembed.W_U": _get_tensor("lm_head.weight", wmap, model_path).T},
    strict=False,
)
model.load_state_dict(
    {"unembed.b_U": torch.zeros(cfg.d_vocab, dtype=cfg.dtype)},
    strict=False,
)
gc.collect()

print(f"\nModel loaded! Layers: {cfg.n_layers}, Experts: {cfg.num_experts}, d_model: {cfg.d_model}")

## Basic Inference

In [ ]:
prompts = [
    "The capital of France is",
    "2 + 2 =",
    "The opposite of hot is",
]

for prompt in prompts:
    tokens = model.to_tokens(prompt)
    with torch.no_grad():
        logits = model(tokens)
    probs = torch.softmax(logits[0, -1].float(), dim=-1)
    top5 = probs.topk(5)
    print(f"Prompt: '{prompt}'")
    for i in range(5):
        token_str = model.to_string(top5.indices[i])
        print(f"  {token_str!r}: {top5.values[i]:.4f}")
    print()

## Caching and Residual Stream

In [ ]:
tokens = model.to_tokens("The cat sat on the mat")
with torch.no_grad():
    logits, cache = model.run_with_cache(tokens)

print("Cached activation keys (first 10):")
for key in list(cache.keys())[:10]:
    print(f"  {key}: {cache[key].shape}")

## Expert Routing Analysis

GPT-OSS routes each token to 4 of 32 experts. We can inspect which experts are selected and their weights.

In [ ]:
str_tokens = model.to_str_tokens("The cat sat on the mat")

for layer in range(min(N_LAYERS, 3)):
    expert_weights = cache[f"blocks.{layer}.mlp.hook_expert_weights"]
    expert_indices = cache[f"blocks.{layer}.mlp.hook_expert_indices"]

    print(f"\nLayer {layer} expert routing:")
    for t in range(len(str_tokens)):
        indices = expert_indices[t].tolist()
        weights = [expert_weights[t, idx].item() for idx in indices]
        pairs = ", ".join(f"E{idx}({w:.2f})" for idx, w in zip(indices, weights))
        print(f"  {str_tokens[t]:>10s} -> {pairs}")

## Logit Lens

Apply the unembedding matrix to intermediate residual streams to see how the model's prediction evolves across layers.

In [ ]:
prompt = "The capital of France is"
tokens = model.to_tokens(prompt)
with torch.no_grad():
    logits, cache = model.run_with_cache(tokens)

print(f"Prompt: '{prompt}'")
print(f"{'Layer':>6s}  {'Top prediction':>20s}  {'Prob':>6s}")
print("-" * 38)

for layer in range(N_LAYERS):
    resid = cache[f"blocks.{layer}.hook_resid_post"]
    normed = model.ln_final(resid)
    layer_logits = normed @ model.unembed.W_U + model.unembed.b_U
    probs = torch.softmax(layer_logits[0, -1].float(), dim=-1)
    top_idx = probs.argmax().item()
    top_word = model.to_string(torch.tensor(top_idx))
    print(f"{layer:>6d}  {top_word:>20s}  {probs[top_idx]:.4f}")

## Attention Patterns

In [ ]:
prompt = "The cat sat on the mat"
tokens = model.to_tokens(prompt)
str_tokens = model.to_str_tokens(prompt)

with torch.no_grad():
    logits, cache = model.run_with_cache(tokens)

# Show attention pattern for layer 0, head 0
pattern = cache["blocks.0.attn.hook_pattern"][0, 0]  # [seq, seq]
print("Layer 0, Head 0 attention pattern:")
print(f"{'':>12s}", "  ".join(f"{t:>6s}" for t in str_tokens))
for i, src_token in enumerate(str_tokens):
    row = "  ".join(f"{pattern[i, j]:.3f}" for j in range(len(str_tokens)))
    print(f"{src_token:>12s}  {row}")

## Activation Patching

Patch the residual stream from a clean run into a corrupted run to measure causal effects.

In [ ]:
clean_prompt = "The capital of France is"
corrupt_prompt = "The capital of Germany is"

clean_tokens = model.to_tokens(clean_prompt)
corrupt_tokens = model.to_tokens(corrupt_prompt)

# Get clean activations
captured_clean = {}

def save_clean(tensor, hook):
    captured_clean[hook.name] = tensor.detach().clone()

with torch.no_grad():
    clean_logits = model.run_with_hooks(
        clean_tokens,
        fwd_hooks=[(f"blocks.{l}.hook_resid_post", save_clean) for l in range(N_LAYERS)],
    )

with torch.no_grad():
    corrupt_logits = model(corrupt_tokens)

clean_pred = model.to_string(clean_logits[0, -1].argmax())
corrupt_pred = model.to_string(corrupt_logits[0, -1].argmax())
print(f"Clean prediction:   '{clean_pred}'")
print(f"Corrupt prediction: '{corrupt_pred}'")

# Patch each layer's residual stream and measure effect
print(f"\n{'Layer':>6s}  {'Patched prediction':>20s}  {'Logit diff':>10s}")
print("-" * 42)

for layer in range(N_LAYERS):
    hook_name = f"blocks.{layer}.hook_resid_post"

    def patch_hook(tensor, hook, clean_act=captured_clean[hook_name]):
        return clean_act

    with torch.no_grad():
        patched_logits = model.run_with_hooks(
            corrupt_tokens,
            fwd_hooks=[(hook_name, patch_hook)],
        )

    patched_pred = model.to_string(patched_logits[0, -1].argmax())
    diff = (patched_logits[0, -1] - corrupt_logits[0, -1]).abs().sum().item()
    print(f"{layer:>6d}  {patched_pred:>20s}  {diff:>10.1f}")